# Edmonton Office Brochure Extraction Pipeline
## Google Colab Entry Point

In [ ]:
# Cell 1 — Instalar dependências
!pip install pymupdf pytesseract Pillow rapidfuzz pydantic openpyxl requests python-dotenv google-genai pandas
!apt-get install -y tesseract-ocr -q

In [ ]:
# Cell 2 — Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Clonar repo e configurar variáveis de ambiente
import os, sys
from google.colab import userdata

REPO_DIR = '/content/brochure-pipeline'

# Sempre parte de /content para evitar aninhamento de pastas
os.chdir('/content')

# Remove clone anterior e clona sempre limpo
!rm -rf {REPO_DIR}
!git clone --branch claude/edmonton-brochure-extraction-2r7Bu \
    https://github.com/dionathan-santos/brochure.git {REPO_DIR}

# Entra no diretório correto (absoluto, nunca relativo)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Chave Gemini (nome do secret no Colab: GOOGLE_MAPS_API_KEY) ───────────
os.environ['GEMINI_API_KEY'] = userdata.get('GOOGLE_MAPS_API_KEY')

# ── Chave Anthropic (fallback opcional) ───────────────────────────────────
try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    os.environ['ANTHROPIC_API_KEY'] = ''

# ── Paths no Google Drive ─────────────────────────────────────────────────
os.environ['MASTER_INVENTORY_PATH'] = '/content/drive/MyDrive/Brochures/Master_Inventory.xlsx'
os.environ['AVAILABILITIES_PATH']   = '/content/drive/MyDrive/Brochures/Availabilities.xlsx'
os.environ['OUTPUT_DIR']            = '/content/drive/MyDrive/Brochures/Output'

# Verificação
key = os.environ.get('GEMINI_API_KEY', '')
print(f'Diretório: {os.getcwd()}')
print(f'GEMINI_API_KEY: {key[:8]}...' if key else '❌ Chave vazia!')
print('✅ Pronto!' if key else '❌ Verifique o nome do secret')

In [ ]:
# Cell 4 — Rodar o pipeline
# Ajuste brochure_dir para a pasta no Drive onde estão os PDFs
# Ajuste brokerage para o nome da corretora (ex: CBRE, Colliers, Cushman)
from main import run_pipeline

output_path = run_pipeline(
    brochure_dir='/content/drive/MyDrive/Brochures/CBRE/',
    brokerage='CBRE'
)
print(f'Output saved: {output_path}')